In [1]:
# ================================================================
# 📅 Grouping days into 30-day periods (Using Sales File Only)
#    No calendar.csv required
# ================================================================

import pandas as pd
import numpy as np
import os

print("="*50)
print("🔥 Grouping Days into 30-Day Periods (No Calendar File)")
print("="*50)

# 1. Base path
BASE_PATH = '/kaggle/input/datasets/aryayadav0513/m5-forecasting-accuracy/m5-forecasting-accuracy/'  

# 2. Identifier columns
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

# 3. Load sales data (Sample of 10,000 rows for fast execution)
print("⏳ Loading sales file...")
sales = pd.read_csv(BASE_PATH + 'sales_train_evaluation.csv', nrows=10000)  

print(f"   ✅ Loaded {sales.shape[0]:,} items × {sales.shape[1]} columns")

# 4. Extract day column names (d_1, d_2, ... d_1941)
day_cols = [col for col in sales.columns if col.startswith('d_')]
print(f"   ✅ Total number of days: {len(day_cols)}")

# 5. Build the new DataFrame with IDs and grouped periods
print("⏳ Grouping days into 30-day periods...")
new_df = sales[id_cols].copy()  

period_number = 1
for i in range(0, len(day_cols), 28):
    # Take 28 columns for each period (or less for the last period)
    cols = day_cols[i:i+28]
    if not cols:
        break
    
    # Sum the values of these days into one column
    new_df[f'Period_{period_number}'] = sales[cols].sum(axis=1)
    period_number += 1

print(f"   ✅ Grouping completed! Total periods: {period_number - 1}")

# 6. Save directly to Excel
output_file = 'sales_every_30_days.xlsx'
new_df.to_excel(output_file, index=False)

print(f"\n✅ File saved successfully to: '{output_file}'")
print(f"📊 Number of items: {new_df.shape[0]:,}")
print(f"📅 Number of periods: {new_df.shape[1] - len(id_cols)}")
print("\n🎯 Done! Open the Excel file to see the grouped periods.")

🔥 Grouping Days into 30-Day Periods (No Calendar File)
⏳ Loading sales file...
   ✅ Loaded 10,000 items × 1947 columns
   ✅ Total number of days: 1941
⏳ Grouping days into 30-day periods...
   ✅ Grouping completed! Total periods: 70

✅ File saved successfully to: 'sales_every_30_days.xlsx'
📊 Number of items: 10,000
📅 Number of periods: 70

🎯 Done! Open the Excel file to see the grouped periods.


In [2]:
# =====================================================================
# 📊 Grouping every 4 rows for each (store_id, item_id) into one row
#    Calculating average sell price for each period
#    Saving results to CSV (or Excel with automatic splitting)
# =====================================================================

import pandas as pd
import os

print("="*60)
print("🔥 Grouping Every 4 Rows into One (Average Price per Period)")
print("="*60)

# ---------- 1. Define Paths and Input File ----------
BASE_PATH = '/kaggle/input/datasets/aryayadav0513/m5-forecasting-accuracy/m5-forecasting-accuracy/'  
INPUT_FILE = 'sell_prices.csv'  

# ---------- 2. Load Data ----------
print("⏳ Loading data file...")
df = pd.read_csv(BASE_PATH + INPUT_FILE)
print(f"   ✅ Loaded {df.shape[0]:,} rows and {df.shape[1]} columns")

# ---------- 3. Check for Required Columns ----------
required = ['store_id', 'item_id', 'sell_price']
for col in required:
    if col not in df.columns:
        raise ValueError(f"❌ Column '{col}' not found. Available columns: {list(df.columns)}")

# ---------- 4. Sort Data by Store and Item ----------
df = df.sort_values(['store_id', 'item_id']).reset_index(drop=True)

# ---------- 5. Create Sequence Number (Starts from 1) ----------
df['seq'] = df.groupby(['store_id', 'item_id']).cumcount() + 1

# ---------- 6. Determine Period Number (Every 4 rows = 1 period) ----------
df['period_num'] = (df['seq'] - 1) // 4 + 1

# ---------- 7. Group Data to Aggregate Each Period ----------
grouped = df.groupby(['store_id', 'item_id', 'period_num'], as_index=False).agg(
    avg_sell_price=('sell_price', 'mean'),
    count=('sell_price', 'count')   # Number of rows in period (might be less than 4 in the last one)
)

# ---------- 8. Add Text Label for Period ----------
grouped['period_label'] = 'Period ' + grouped['period_num'].astype(str)

# ---------- 9. Arrange Final Columns ----------
output_cols = ['store_id', 'item_id', 'period_label', 'avg_sell_price', 'count']
result = grouped[output_cols]

# ---------- 10. Display Summary Statistics ----------
print(f"\n📊 Original row count: {df.shape[0]:,}")
print(f"📊 Row count after grouping (Every 4 rows → 1 row): {result.shape[0]:,}")
print(f"   (Approximately 25% of original size, accounting for incomplete periods)")

# ---------- 11. Save Results ----------

# Option 1: Save as CSV (Recommended for large datasets)
output_csv = 'output_one_row_per_period.csv'
result.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"\n✅ File saved successfully as CSV to: '{output_csv}'")

# Option 2: Save as Excel with automatic sheet splitting (Each sheet <= 1,000,000 rows)
# Uncomment the block below if Excel output is needed
"""
output_excel = 'output_one_row_per_period.xlsx'
chunk_size = 1_000_000
num_chunks = (len(result) + chunk_size - 1) // chunk_size

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, len(result))
        chunk = result.iloc[start:end]
        sheet_name = f'Part_{i+1}'
        chunk.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"   ✅ Written sheet {sheet_name} (rows {start+1}-{end})")
print(f"\n✅ File saved successfully as Excel to: '{output_excel}'")
"""

# ---------- 12. Display Sample Results ----------
print("\n🎯 Sample of results (First 10 rows):")
print(result.head(10))

# ---------- 13. Explanatory Example ----------
print("\n📌 Example: First 3 periods for item (store_id=1, item_id=100):")
sample = result[(result['store_id'] == 1) & (result['item_id'] == 100)].head(3)
print(sample)

print("\n✅ Process completed successfully!")

🔥 Grouping Every 4 Rows into One (Average Price per Period)
⏳ Loading data file...
   ✅ Loaded 6,841,121 rows and 4 columns

📊 Original row count: 6,841,121
📊 Row count after grouping (Every 4 rows → 1 row): 1,723,255
   (Approximately 25% of original size, accounting for incomplete periods)

✅ File saved successfully as CSV to: 'output_one_row_per_period.csv'

🎯 Sample of results (First 10 rows):
  store_id      item_id period_label  avg_sell_price  count
0     CA_1  FOODS_1_001     Period 1             2.0      4
1     CA_1  FOODS_1_001     Period 2             2.0      4
2     CA_1  FOODS_1_001     Period 3             2.0      4
3     CA_1  FOODS_1_001     Period 4             2.0      4
4     CA_1  FOODS_1_001     Period 5             2.0      4
5     CA_1  FOODS_1_001     Period 6             2.0      4
6     CA_1  FOODS_1_001     Period 7             2.0      4
7     CA_1  FOODS_1_001     Period 8             2.0      4
8     CA_1  FOODS_1_001     Period 9             2.0      4

In [3]:
import pandas as pd

# 1. Read data from the specified Kaggle path
file_path = '/kaggle/input/datasets/hishamkhalidbushaar/sales-every-30-days/sales_every_30_days (1).xlsx'
df_sales = pd.read_excel(file_path, sheet_name='Sheet1')
df_prices = pd.read_excel(file_path, sheet_name='Sheet2')

print("...Starting period cleaning and guaranteed merge...")

# 2. Melt period columns from the first sheet into rows
id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
df_sales_long = df_sales.melt(
    id_vars=id_vars, 
    var_name='Period', 
    value_name='Quantity'
)

# 3. Clean period names in both dataframes (remove spaces to ensure exact match)
# This standardizes the period format in both sheets (e.g., Period_1)
df_sales_long['Period'] = df_sales_long['Period'].str.replace(' ', '').str.strip()
df_prices['Period'] = df_prices['Period'].str.replace(' ', '').str.strip()

# 4. Merge based on exact store_id, item_id, and Period
kaggle_dataset = pd.merge(
    df_prices, 
    df_sales_long, 
    on=['store_id', 'item_id', 'Period'], 
    how='inner'
)

# 5. Display results to verify the merged dataset
print(f"Successfully merged rows: {len(kaggle_dataset)}")
print(kaggle_dataset.head())

# 6. Save the final file
kaggle_dataset.to_csv('merged_sales_data.csv', index=False)
print("\nFile saved successfully with complete data as: merged_sales_data.csv")

...Starting period cleaning and guaranteed merge...
Successfully merged rows: 553692
  store_id      item_id  avg_sell_price    Period  \
0     CA_1  FOODS_1_001             2.0  Period_1   
1     CA_1  FOODS_1_001             2.0  Period_2   
2     CA_1  FOODS_1_001             2.0  Period_3   
3     CA_1  FOODS_1_001             2.0  Period_4   
4     CA_1  FOODS_1_001             2.0  Period_5   

                            id  dept_id cat_id state_id  Quantity  
0  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        39  
1  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        39  
2  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        24  
3  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        36  
4  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        32  

File saved successfully with complete data as: merged_sales_data.csv


In [4]:
import pandas as pd

# 1. Define the correct file path for merged_sales_data.csv in Kaggle
file_path = '/kaggle/input/datasets/hishamkhalidbushaar/merged-sales-data/merged_sales_data.csv'

print(f"Reading file from path: {file_path}")
df = pd.read_csv(file_path)

# Strip any leading/trailing spaces from column names
df.columns = df.columns.str.strip()

# Find the period column regardless of case (Period or period) to prevent errors
period_col = [col for col in df.columns if col.lower() == 'period']
if not period_col:
    raise KeyError("Period column not found. Please verify column names in your file.")
else:
    actual_period_name = period_col[0]

# 2. Extract the period number and convert it to an integer
df['period_num'] = df[actual_period_name].astype(str).str.extract(r'(\d+)').astype(int)

# 3. Define the official start date for the M5 dataset (January 29, 2011)
start_date = pd.to_datetime('2011-01-29')

# 4. Calculate start and end dates for each period (incrementing month by month)
df['Start_Date'] = df['period_num'].apply(lambda x: start_date + pd.DateOffset(months=x-1))
df['End_Date'] = df['period_num'].apply(lambda x: start_date + pd.DateOffset(months=x) - pd.DateOffset(days=1))

# 5. Drop the helper column
df.drop(columns=['period_num'], inplace=True)

# 6. Preview the results to verify dates
print("\n--- Preview of the first 5 rows with converted dates ---")
print(df.head())

print("\n--- Period sequence and converted dates ---")
print(df[[actual_period_name, 'Start_Date', 'End_Date']].drop_duplicates().head(12))

# 7. Save the final file in Kaggle output directory
output_file = 'sales_data_with_dates.csv'
df.to_csv(output_file, index=False)
print(f"\nFile saved successfully to output directory as: {output_file}")

Reading file from path: /kaggle/input/datasets/hishamkhalidbushaar/merged-sales-data/merged_sales_data.csv

--- Preview of the first 5 rows with converted dates ---
  store_id      item_id  avg_sell_price    Period  \
0     CA_1  FOODS_1_001             2.0  Period_1   
1     CA_1  FOODS_1_001             2.0  Period_2   
2     CA_1  FOODS_1_001             2.0  Period_3   
3     CA_1  FOODS_1_001             2.0  Period_4   
4     CA_1  FOODS_1_001             2.0  Period_5   

                            id  dept_id cat_id state_id  Quantity Start_Date  \
0  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        39 2011-01-29   
1  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        39 2011-02-28   
2  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        24 2011-03-29   
3  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        36 2011-04-29   
4  FOODS_1_001_CA_1_evaluation  FOODS_1  FOODS       CA        32 2011-05-29   

    End_Date  
0 2011-02-27  
1 201

In [7]:
# =============================================================
# 📈 Exploratory Data Analysis (EDA)
# =============================================================
print("\n" + "="*50)
print("📊 Running Exploratory Data Analysis (EDA)...")
print("="*50)

# 1. Check dataset general shape and missing values
print(f"🔹 Dataset Dimensions: {df.shape[0]:,} rows | {df.shape[1]} columns")
print("\n🔹 Missing Values per Column:")
print(df.isnull().sum())

# 2. Key statistical insights for price and quantity
print("\n🔹 Numerical Features Distribution Summary:")
print(df[['Quantity', 'avg_sell_price']].describe())

# 3. Analyze volume distribution across categories
print("\n🔹 Total Sales Volume by Product Category:")
cat_sales = df.groupby('cat_id')['Quantity'].sum().reset_index()
print(cat_sales)

# 4. Analyze price variance to prove dynamic pricing existence
print("\n🔹 Price Variance Analysis (Dynamic vs Static Pricing):")
print(price_variance['Is_Dynamic_Price'].value_counts().rename({1: 'Dynamic Price Items', 0: 'Static Price Items'}))
print("="*50 + "\n")


📊 Running Exploratory Data Analysis (EDA)...
🔹 Dataset Dimensions: 553,692 rows | 15 columns

🔹 Missing Values per Column:
store_id          0
item_id           0
avg_sell_price    0
Period            0
id                0
dept_id           0
cat_id            0
state_id          0
Quantity          0
Start_Date        0
End_Date          0
Last Quantity     0
Last price        0
Month             0
Year              0
dtype: int64

🔹 Numerical Features Distribution Summary:
            Quantity  avg_sell_price
count  553692.000000   553692.000000
mean       40.111493        4.489755
std       111.447610        3.531796
min         0.000000        0.100000
25%         0.000000        2.180000
50%        12.000000        3.480000
75%        38.000000        5.880000
max      9373.000000       30.980000

🔹 Total Sales Volume by Product Category:
      cat_id  Quantity
0      FOODS  14598276
1    HOBBIES   2674421
2  HOUSEHOLD   4936716

🔹 Price Variance Analysis (Dynamic vs Static Prici

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score
import os

print("🔍 Reading data and calculating accuracy levels for the three categories...")

# 1. Read file from Kaggle
file_path = '/kaggle/input/datasets/hishamkhalidbushaar/sales-data-with-dates/sales_data_with_dates.csv'
try:
    df = pd.read_csv(file_path)
except FileNotFoundError:
    file_path_found = None
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            if 'sales_data_with_dates.csv' in filename:
                file_path_found = os.path.join(dirname, filename)
                break
    df = pd.read_csv(file_path_found)

df.columns = df.columns.str.strip()

# Prepare dates and time columns
df['Start_Date'] = pd.to_datetime(df['Start_Date'], dayfirst=True, errors='coerce')
df['Month'] = df['Start_Date'].dt.month.fillna(1).astype(int)
df['Year'] = df['Start_Date'].dt.year.fillna(2016).astype(int)

df['Last Quantity'] = df['Last Quantity'].fillna(df['Quantity'])
df['Last price'] = df['Last price'].fillna(df['avg_sell_price'])
df = df.sort_values(by=['item_id', 'store_id', 'Year', 'Month']).reset_index(drop=True)

# Filter dynamic price items to capture clear price elasticity
price_variance = df.groupby(['store_id', 'item_id'])['avg_sell_price'].nunique().reset_index()
price_variance['Is_Dynamic_Price'] = np.where(price_variance['avg_sell_price'] > 1, 1, 0)
df = df.merge(price_variance[['store_id', 'item_id', 'Is_Dynamic_Price']], on=['store_id', 'item_id'], how='left')

# =============================================================
# 📊 Part 1: Price Impact Model Accuracy per Category
# =============================================================
df_dynamic = df[df['Is_Dynamic_Price'] == 1].copy()

df_dynamic['Rolling_Qty_2Period'] = df_dynamic.groupby(['store_id', 'item_id'])['Last Quantity'].transform(lambda x: x.rolling(window=2, min_periods=1).mean())
df_dynamic['Rolling_Qty_3Period'] = df_dynamic.groupby(['store_id', 'item_id'])['Last Quantity'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df_dynamic['Quantity_Velocity'] = df_dynamic['Last Quantity'] - df_dynamic['Rolling_Qty_2Period']

df_dynamic['Price_Ratio'] = df_dynamic['avg_sell_price'] / (df_dynamic['Last price'] + 1e-5)
df_dynamic['Price_Diff'] = df_dynamic['avg_sell_price'] - df_dynamic['Last price']
df_dynamic['Item_Store_Mean_Price'] = df_dynamic.groupby(['store_id', 'item_id'])['avg_sell_price'].transform('mean')
df_dynamic['Price_vs_Historical_Mean'] = df_dynamic['avg_sell_price'] / (df_dynamic['Item_Store_Mean_Price'] + 1e-5)
df_dynamic['Month_Sin'] = np.sin(2 * np.pi * df_dynamic['Month'] / 12)
df_dynamic['Price_Season_Interaction'] = df_dynamic['avg_sell_price'] * df_dynamic['Month_Sin']

le_cat = LabelEncoder()
df_dynamic['cat_id_encoded'] = le_cat.fit_transform(df_dynamic['cat_id'].astype(str))
df_dynamic['Store_Cat_Avg_Price'] = df_dynamic.groupby(['store_id', 'cat_id_encoded'])['avg_sell_price'].transform('mean')
df_dynamic['Price_Competitiveness'] = df_dynamic['avg_sell_price'] / (df_dynamic['Store_Cat_Avg_Price'] + 1e-5)
df_dynamic['Price_Quantity_Interaction'] = df_dynamic['avg_sell_price'] * df_dynamic['Last Quantity']
df_dynamic['Expected_Zad'] = df_dynamic['Last Quantity'] / (df_dynamic['Price_Ratio'] + 1e-5)

le_store = LabelEncoder()
df_dynamic['store_id_encoded'] = le_store.fit_transform(df_dynamic['store_id'].astype(str))

features_price = [
    'cat_id_encoded', 'store_id_encoded', 'avg_sell_price', 'Month', 
    'Last Quantity', 'Last price', 'Price_Ratio', 'Price_Diff', 
    'Price_vs_Historical_Mean', 'Price_Season_Interaction', 'Price_Quantity_Interaction',
    'Price_Competitiveness', 'Expected_Zad', 
    'Rolling_Qty_2Period', 'Rolling_Qty_3Period', 'Quantity_Velocity'
]

df_train_p = df_dynamic.dropna(subset=['avg_sell_price', 'Quantity']).copy()
X_p = df_train_p[features_price]
y_p = df_train_p['Quantity']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

model_price = xgb.XGBRegressor(n_estimators=300, max_depth=9, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1)
model_price.fit(X_train_p, y_train_p)
y_pred_p = model_price.predict(X_test_p)

test_p_analysis = X_test_p.copy()
test_p_analysis['Actual'] = y_test_p
test_p_analysis['Pred'] = y_pred_p
test_p_analysis['cat_id'] = le_cat.inverse_transform(test_p_analysis['cat_id_encoded'])

# Calculate overall metrics for price model
mae_p_total = np.mean(np.abs(y_test_p - y_pred_p))
accuracy_price_total = (1 - (mae_p_total / np.mean(y_test_p))) * 100

# =============================================================
# 🎯 Part 2: Demand Forecasting Model Accuracy per Category
# =============================================================
df_combined = df.copy()
df_combined['Lag_1_Month'] = df_combined.groupby(['item_id', 'store_id'])['Last Quantity'].shift(1).fillna(0)
df_combined['Lag_2_Months'] = df_combined.groupby(['item_id', 'store_id'])['Last Quantity'].shift(2).fillna(0)
df_combined['Lag_12_Months'] = df_combined.groupby(['item_id', 'store_id'])['Last Quantity'].shift(12).fillna(0)
df_combined['Rolling_Mean_3_Months'] = df_combined.groupby(['item_id', 'store_id'])['Last Quantity'].transform(lambda x: x.rolling(3, min_periods=1).mean()).fillna(0)
df_combined['Rolling_Mean_6_Months'] = df_combined.groupby(['item_id', 'store_id'])['Last Quantity'].transform(lambda x: x.rolling(6, min_periods=1).mean()).fillna(0)

categorical_cols = ['store_id', 'item_id', 'dept_id', 'cat_id', 'state_id']
df_encoded_f = df_combined.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded_f[col] = le.fit_transform(df_encoded_f[col].astype(str)).astype('int')

df_encoded_f['store_id_encoded'] = df_encoded_f['store_id']
df_encoded_f['cat_id_encoded'] = df_encoded_f['cat_id']

features_forecast = ['store_id_encoded', 'item_id', 'dept_id', 'cat_id_encoded', 'state_id', 
                     'avg_sell_price', 'Last price', 'Last Quantity', 'Month', 'Year', 
                     'Lag_1_Month', 'Lag_2_Months', 'Lag_12_Months', 'Rolling_Mean_3_Months', 'Rolling_Mean_6_Months']

X_f = df_encoded_f[features_forecast]
y_f = df_encoded_f['Quantity']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

model_forecast = xgb.XGBRegressor(n_estimators=300, max_depth=9, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1)
model_forecast.fit(X_train_f, y_train_f)
y_pred_f = model_forecast.predict(X_test_f)

test_f_analysis = X_test_f.copy()
test_f_analysis['Actual'] = y_test_f
test_f_analysis['Pred'] = y_pred_f
test_f_analysis['cat_id'] = df_combined.loc[X_test_f.index, 'cat_id'].values

# =============================================================
# 📋 Print Detailed Accuracy Report for the Three Categories
# =============================================================
print("\n" + "="*70)
print("📊 Accuracy Report for the Two Models across Three Categories")
print("="*70)
print(f"1️⃣ Price Impact Model (Elasticity based on Price) | Overall Accuracy: {accuracy_price_total:.2f}%")
for cat in test_p_analysis['cat_id'].unique():
    cat_df = test_p_analysis[test_p_analysis['cat_id'] == cat]
    cat_mae = np.mean(np.abs(cat_df['Actual'] - cat_df['Pred']))
    cat_acc = (1 - (cat_mae / np.mean(cat_df['Actual']))) * 100
    print(f"  - Category [{str(cat):<10}]: Price impact accuracy is {cat_acc:.2f}%")

print("-"*70)
print("2️⃣ Demand Forecasting Model (Seasonality and Time)")
for cat in test_f_analysis['cat_id'].unique():
    cat_df = test_f_analysis[test_f_analysis['cat_id'] == cat]
    cat_acc = r2_score(cat_df['Actual'], cat_df['Pred']) * 100
    print(f"  - Category [{str(cat):<10}]: Demand forecasting accuracy is {cat_acc:.2f}%")
print("="*70 + "\n")

# =============================================================
# 📅 Create 2017 Grid and Generate Scenarios
# =============================================================
print("⚙️ Generating 2017 scenarios and fixing economic curve slopes...")
base_combinations = df_train_p[['store_id', 'item_id', 'dept_id', 'cat_id', 'state_id', 'cat_id_encoded', 'store_id_encoded']].drop_duplicates()
months_2017 = pd.DataFrame({'Month': range(1, 13), 'Year': 2017})
df_2017_scenarios = base_combinations.assign(key=1).merge(months_2017.assign(key=1), on='key').drop(columns='key')

last_history = df_train_p.sort_values(by=['Year', 'Month']).groupby(['item_id', 'store_id']).last().reset_index()
df_2017_scenarios = df_2017_scenarios.merge(
    last_history[['item_id', 'store_id', 'avg_sell_price', 'Last price', 'Rolling_Qty_2Period', 'Rolling_Qty_3Period', 'Quantity_Velocity', 'Last Quantity', 'Item_Store_Mean_Price', 'Store_Cat_Avg_Price']], 
    on=['item_id', 'store_id'], how='left'
)

df_2017_scenarios['Item_Store_Mean_Price'] = df_2017_scenarios['Item_Store_Mean_Price'].fillna(df_2017_scenarios['avg_sell_price'])
df_2017_scenarios['Store_Cat_Avg_Price'] = df_2017_scenarios['Store_Cat_Avg_Price'].fillna(df_2017_scenarios['avg_sell_price'])
df_2017_scenarios['Month_Sin'] = np.sin(2 * np.pi * df_2017_scenarios['Month'] / 12)

# Feature engineering for 2017 scenarios
df_2017_scenarios['Price_Ratio'] = df_2017_scenarios['avg_sell_price'] / (df_2017_scenarios['Last price'] + 1e-5)
df_2017_scenarios['Price_Diff'] = df_2017_scenarios['avg_sell_price'] - df_2017_scenarios['Last price']
df_2017_scenarios['Price_vs_Historical_Mean'] = df_2017_scenarios['avg_sell_price'] / (df_2017_scenarios['Item_Store_Mean_Price'] + 1e-5)
df_2017_scenarios['Price_Season_Interaction'] = df_2017_scenarios['avg_sell_price'] * df_2017_scenarios['Month_Sin']
df_2017_scenarios['Price_Competitiveness'] = df_2017_scenarios['avg_sell_price'] / (df_2017_scenarios['Store_Cat_Avg_Price'] + 1e-5)
df_2017_scenarios['Price_Quantity_Interaction'] = df_2017_scenarios['avg_sell_price'] * df_2017_scenarios['Last Quantity']
df_2017_scenarios['Expected_Zad'] = df_2017_scenarios['Last Quantity'] / (df_2017_scenarios['Price_Ratio'] + 1e-5)

base_price = df_2017_scenarios['avg_sell_price'].copy()

scenarios = {
    'Base': 1.0,
    'Price_Up_5pct': 1.05,
    'Price_Up_10pct': 1.10,
    'Price_Down_5pct': 0.95,
    'Price_Down_10pct': 0.90
}

# Final fit of the price model on all historical data before 2017 forecasting
model_price.fit(X_p, y_p)

# Predict Base Forecast for 2017
df_2017_scenarios['Forecast_Base'] = model_price.predict(df_2017_scenarios[features_price])

for name, multiplier in scenarios.items():
    if name == 'Base':
        df_2017_scenarios[f'Forecast_{name}'] = df_2017_scenarios['Forecast_Base']
        continue
        
    df_grid_scen = df_2017_scenarios.copy()
    df_grid_scen['avg_sell_price'] = base_price * multiplier
    
    df_grid_scen['Price_Ratio'] = df_grid_scen['avg_sell_price'] / (df_grid_scen['Last price'] + 1e-5)
    df_grid_scen['Price_Diff'] = df_grid_scen['avg_sell_price'] - df_grid_scen['Last price']
    df_grid_scen['Price_vs_Historical_Mean'] = df_grid_scen['avg_sell_price'] / (df_grid_scen['Item_Store_Mean_Price'] + 1e-5)
    df_grid_scen['Price_Season_Interaction'] = df_grid_scen['avg_sell_price'] * df_grid_scen['Month_Sin']
    df_grid_scen['Price_Competitiveness'] = df_grid_scen['avg_sell_price'] / (df_grid_scen['Store_Cat_Avg_Price'] + 1e-5)
    df_grid_scen['Price_Quantity_Interaction'] = df_grid_scen['avg_sell_price'] * df_grid_scen['Last Quantity']
    df_grid_scen['Expected_Zad'] = df_grid_scen['Last Quantity'] / (df_grid_scen['Price_Ratio'] + 1e-5)
    
    pred_values = model_price.predict(df_grid_scen[features_price])
    
    # Enforce strict economic laws (Negative elasticity constraint)
    if 'Up' in name:
        df_2017_scenarios[f'Forecast_{name}'] = np.minimum(pred_values, df_2017_scenarios['Forecast_Base'])
    elif 'Down' in name:
        df_2017_scenarios[f'Forecast_{name}'] = np.maximum(pred_values, df_2017_scenarios['Forecast_Base'])

# Calculate row-level errors historically at the store-item level
df_train_p['Row_Error'] = np.where(df_train_p['Quantity'] > 0, np.abs(df_train_p['Quantity'] - model_price.predict(X_p)) / df_train_p['Quantity'], 0)
item_error_map = df_train_p.groupby(['store_id', 'item_id'])['Row_Error'].mean().reset_index()
item_error_map.rename(columns={'Row_Error': 'Item_Store_Mean_Error'}, inplace=True)

df_2017_scenarios = df_2017_scenarios.merge(item_error_map, on=['store_id', 'item_id'], how='left')
df_2017_scenarios['Item_Store_Mean_Error'] = df_2017_scenarios['Item_Store_Mean_Error'].fillna(0.15)

df_2017_scenarios['Safety_Upper_Bound'] = df_2017_scenarios['Forecast_Base'] * (1 + df_2017_scenarios['Item_Store_Mean_Error'])
df_2017_scenarios['Safety_Lower_Bound'] = df_2017_scenarios['Forecast_Base'] * (1 - df_2017_scenarios['Item_Store_Mean_Error'])
df_2017_scenarios['Safety_Lower_Bound'] = df_2017_scenarios['Safety_Lower_Bound'].clip(lower=0)

# Filter columns and save output dataset
final_pbi_cols = [
    'Month', 'Year', 'store_id', 'item_id', 'cat_id', 'avg_sell_price', 'Item_Store_Mean_Error',
    'Forecast_Base', 'Forecast_Price_Up_5pct', 'Forecast_Price_Up_10pct', 'Forecast_Price_Down_5pct', 'Forecast_Price_Down_10pct',
    'Safety_Upper_Bound', 'Safety_Lower_Bound'
]
df_final_pbi = df_2017_scenarios[[col for col in final_pbi_cols if col in df_2017_scenarios.columns]]
df_final_pbi.to_csv('hisham_2017_price_simulation_dashboard.csv', index=False)

print("🎯 Completed and saved the main dataset successfully!")
print("📁 Output dashboard filename: (hisham_2017_price_simulation_dashboard.csv)")
print("="*70)

🔍 Reading data and calculating accuracy levels for the three categories...

📊 Accuracy Report for the Two Models across Three Categories
1️⃣ Price Impact Model (Elasticity based on Price) | Overall Accuracy: 69.33%
  - Category [HOUSEHOLD ]: Price impact accuracy is 68.84%
  - Category [FOODS     ]: Price impact accuracy is 69.40%
  - Category [HOBBIES   ]: Price impact accuracy is 69.71%
----------------------------------------------------------------------
2️⃣ Demand Forecasting Model (Seasonality and Time)
  - Category [FOODS     ]: Demand forecasting accuracy is 89.83%
  - Category [HOUSEHOLD ]: Demand forecasting accuracy is 88.92%
  - Category [HOBBIES   ]: Demand forecasting accuracy is 88.41%

⚙️ Generating 2017 scenarios and fixing economic curve slopes...
🎯 Completed and saved the main dataset successfully!
📁 Output dashboard filename: (hisham_2017_price_simulation_dashboard.csv)
